# 4. Feature Engineering

Après le nettoyage des données, nous allons examiner les variables et effectuer quelques transformations pour mieux représenter les caractéristiques des clients. L'objectif est de simplifier certaines catégories et de créer des indicateurs utiles à la prédiction de la souscription bancaire. Les transformations retenues seront vérifiées au fur et à mesure et pourront être ajustées si nécessaire.

### 4.1. Chargement des données

Nous repartons du fichier nettoyé lors de l'étape précédente afin de conserver les corrections déjà effectuées. Le fichier brut reste inchangé et servira de référence si nous devons revenir sur certains choix.

In [1]:
import pandas as pd

# Chargement des données nettoyées
df = pd.read_csv(
    "../data/interim/bank_marketing_clean.csv",
    sep=";"
)

print("Dimensions du jeu de données :", df.shape)
df.head()

Dimensions du jeu de données : (4119, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,no,no,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


### 4.2. Regroupement des niveaux d’éducation

La variable `education` comporte plusieurs modalités correspondant à des niveaux d’études proches. Pour simplifier son utilisation dans les analyses et les futurs modèles, nous regroupons les modalités `basic.4y`, `basic.6y`, `basic.9y` et `illiterate` dans une seule catégorie appelée `basic`.

In [2]:
# Répartition initiale des niveaux d'éducation
print(df["education"].value_counts(dropna=False))

education
university.degree      1264
high.school             921
basic.9y                574
professional.course     535
basic.4y                429
basic.6y                228
unknown                 167
illiterate                1
Name: count, dtype: int64


In [3]:
# Regroupement des niveaux d'éducation
df["education"] = df["education"].replace({
    "basic.4y": "basic",
    "basic.6y": "basic",
    "basic.9y": "basic",
    "illiterate": "basic"
})

# Vérification après regroupement
print(df["education"].value_counts(dropna=False))

education
university.degree      1264
basic                  1232
high.school             921
professional.course     535
unknown                 167
Name: count, dtype: int64


### 4.3. Regroupement des catégories professionnelles

La variable `job` comporte plusieurs catégories professionnelles. Certaines peuvent être regroupées afin de simplifier les analyses et de faciliter l’interprétation des résultats. Nous allons d’abord examiner les effectifs de chaque catégorie avant de procéder aux transformations.

In [4]:
# Répartition des catégories professionnelles
print(df["job"].value_counts(dropna=False))

job
admin.           1012
blue-collar       884
technician        691
services          393
management        324
retired           166
self-employed     159
entrepreneur      148
unemployed        111
housemaid         110
student            82
unknown            39
Name: count, dtype: int64


In [5]:
# Regroupement des catégories professionnelles
regroupement_job = {
    "admin.": "white_collar",
    "management": "white_collar",
    "blue-collar": "worker",
    "technician": "worker",
    "services": "worker",
    "entrepreneur": "self_employed",
    "self-employed": "self_employed",
    "retired": "inactive",
    "student": "inactive",
    "unemployed": "unemployed"
}

# Application du regroupement
df["job"] = df["job"].map(regroupement_job).fillna("other")

# Vérification des effectifs après regroupement
print(df["job"].value_counts(dropna=False))

job
worker           1968
white_collar     1336
self_employed     307
inactive          248
other             149
unemployed        111
Name: count, dtype: int64
